# C2 No-Progress / Time-Decay Exit — Phase 1

NP50 is the sole formal primary (50% planned duration, prior completed-M1 MFE < +0.25R). NP75 is robustness. 2022–2026 is previously viewed, not a pristine holdout. Total R is primary; no money simulation. EA/SET/RunId/VPS/live and Dell Phase 5 Global R2 remain unchanged.

Plan SHA: `1b211146d4aa7610df9663693e586890d4dba4fe`. Read `docs/95_c2_no_progress_time_decay_phase1_plan.md` before running. Missing inputs or R0 mismatches halt execution.

In [ ]:
from pathlib import Path
import sys, subprocess, json, shutil
import pandas as pd
from IPython.display import display
REPO = Path('/content/time-entry-portfolio-lab')
BASELINE = Path('/content/daily_stop_baseline_trades.csv')
M1_ROOT = Path('/content/m1')
OUT = Path('/content/c2_no_progress_phase1')
SAVE_TO_DRIVE = False
DRIVE_DEST = Path('/content/drive/MyDrive/time-entry-portfolio-lab/c2_no_progress_phase1')
# Supply this existing repository (branch research/c2-no-progress-time-decay-phase1)
# and the exact baseline/56 source files. No Drive mount or write by default.
assert REPO.is_dir() and BASELINE.is_file() and M1_ROOT.is_dir()
record = REPO/'results/c2_no_progress_phase1/c2_no_progress_phase1_run_record.csv'
IMPLEMENTATION_SHA = str(pd.read_csv(record).iloc[0]['ImplementationSHA']) if record.exists() else subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
# Verify that calculation and validation sources match the frozen implementation.
for relative in ['src/research/c2_no_progress_phase1.py','tests/verify_c2_no_progress_phase1.py','src/research/c1_path_management_phase1.py','src/research/exit_efficiency_phase1.py','src/research/daily_stop_baseline_revalidation.py']:
    frozen = subprocess.check_output(['git','-C',str(REPO),'show',f'{IMPLEMENTATION_SHA}:{relative}'])
    assert (REPO/relative).read_bytes() == frozen, relative
print('Implementation:', IMPLEMENTATION_SHA)


In [ ]:
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(REPO/'tests'),'-p','test_c2_no_progress_phase1.py','-v'],check=True)
args = ['--baseline',str(BASELINE),'--manifest',str(REPO/'results/volatility_phase1/volatility_phase1_input_manifest.csv'),'--m1-root',str(M1_ROOT),'--out',str(OUT)]
subprocess.run([sys.executable,str(REPO/'src/research/c2_no_progress_phase1.py'),*args,'--implementation-sha',IMPLEMENTATION_SHA],check=True)
subprocess.run([sys.executable,str(REPO/'tests/verify_c2_no_progress_phase1.py'),*args],check=True)
def table(name):
    return pd.read_csv(OUT/f'c2_no_progress_phase1_{name}.csv')


## 1. Source / Hash Audit

In [ ]:
display(table('m1_audit'))

## 2. R0 Reconciliation

In [ ]:
display(table('r0_reconciliation'))

## 3. NP50 Trigger Coverage

In [ ]:
display(table('coverage'))

## 4. R0 vs NP50 Portfolio

In [ ]:
display(table('portfolio_summary'))

## 5. Historical / Recent Results

In [ ]:
display(table('period_summary'))

## 6. Paired Delta 95% CI

In [ ]:
display(table('trade_delta_summary'))

## 7. Triggered Trade Results

In [ ]:
display(table('trigger_summary'))

## 8. Recovery Diagnostic

In [ ]:
display(table('recovery_summary'))

## 9. NP75 Robustness

In [ ]:
display(table('np75_robustness'))

## 10. Strategy-Level Diagnostic

In [ ]:
display(table('strategy_summary'))

## 11. Formal Gate A–H

In [ ]:
display(table('formal_gates'))

## 12. Final Verdict / 13. Phase 2 Eligibility

In [ ]:
display(table('run_record'))

## Validation and Independent Representative Calculations

In [ ]:
display(table('independent_verification'))

Strategy-specific positive signals are exploratory only. Recovery categories overlap; NEVER_RECOVERED denotes a nonpositive final baseline result, not absence of a temporary recovery. Worst Day/Week use entry dates; MaxDD is cumulative realized R, not mark-to-market or money drawdown. No Phase 2 is performed in this notebook.

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DEST.mkdir(parents=True, exist_ok=True)
    for file in OUT.glob('*.csv'):
        shutil.copy2(file, DRIVE_DEST/file.name)
else:
    print('Drive save OFF; results retained under', OUT)
